# B2.1 · What building a harness means in security engineering

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *Both directions*

Builds on **[B2.0 · The AI SDLC — what runs before a deploy, and what runs after](https://spbreed.github.io/cyber-commons/lessons/B2.0.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** What a harness is — the wrapper around a model that turns
generating text into getting work done — and the four moves of its loop, run
against a real CyberTravels finding with an actual LLM call in it.

**Why a security engineer needs it.** Every part of that wrapper is a security
decision: the tool surface is an authorisation problem, the verifier decides
what the pipeline is allowed to believe, and the budget is the only control
still standing once the model is the component you cannot trust. A harness whose
verifier is the model agreeing with itself does not fail loudly — it succeeds
incorrectly and files a clean trace.

This is a **mechanism** lesson, and it is short on purpose: three examples of
harnesses you already run, then one loop of about twenty lines, executed twice.

## 1 · The hook

A model on its own is a text generator. Wrap it in a loop with tools and it reviews CyberTravels' pull requests; wrap it badly and it reviews them and tells you it found nothing. Every part of that wrapper is a security decision, and nobody else in the building is going to notice that the verifier is a shape check.

> **At CyberTravels.** Alex is building the reviewer that reads CyberTravels' pull requests. Its verifier is the part that decides whether it found anything, and a verifier that asks the model whether it is happy reports a clean review of a vulnerable diff.

## 2 · The framework

```
   a model                 a harness

   tokens in               PLAN   the model proposes
   tokens out         ->   ACT    the harness runs it against a tool
   no memory               VERIFY something independent decides
   no actions              STOP   verified, or a budget ran out
   no notion of
   "did that work"         the security decisions live in the last two

   no verifier   -> the loop accepts whatever came back
   a shape check -> it accepts anything well-formed
   an LLM judge  -> it accepts anything confident
   a real check  -> it refuses escape(ref) because that is still
                    concatenation
```

**Building a harness, in security engineering, is building everything around a
model that turns generating text into getting work done: what it sees, what it
may do, how you know whether it worked, and when it stops.** A model on its own
is a text generator — no memory, no actions, no notion of success. The wrapper
is what makes it a control, and every part of that wrapper is a security
decision: the tool surface is an authorisation problem, the verifier decides
what your pipeline is allowed to believe, and the budget is the only thing
still holding once the model is the component you cannot trust.

You already run several, whether or not anyone calls them that:

- **The CI security scan.** Sees the changed files, may only read, "worked"
  means a zero exit and a finding that parses, stops on a timeout.
- **An autofix bot.** Sees the finding and the file, may open a branch, and
  "worked" should mean *the exploit no longer reproduces* — not *the scanner
  went quiet*.
- **A SOC triage assistant.** Sees the alert and its enrichment, may query the
  SIEM read-only, and "worked" means its disposition matched an analyst's on a
  held-out sample.

The third field is where harnesses fail, and they fail quietly. A harness whose
verifier is the model agreeing with itself does not stop and report an error —
it **succeeds incorrectly**, files a clean trace, and the bug is found later by
whoever merged the patch.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">move</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">who does it</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the security decision in it</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>plan</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the model</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">none — this is the part you did not write</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>act</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the harness</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">the tool surface: what is even expressible</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>verify</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">something independent</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>what the pipeline is allowed to believe</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>stop</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the harness</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">the budget — the last control still standing</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Frameworks make plan and act easy and leave verify and stop as your problem, usually defaulting to “the model says it is done” and “loop forever”.</div>

## 3 · One loop, run twice

The simplest harness that shows the point: about twenty lines, one real model call, one SQL injection from CyberTravels' booking service. It runs with no verifier and then with one — same model, same prompt.

Offline the model is a labelled replay; against a served open-weight endpoint it is the identical code.

### The skill — [`skills/appsec/agentic-harness-loop/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/agentic-harness-loop/SKILL.md)

```yaml
name: agentic-harness-loop
description: >-
  Build and check the plan-act-verify loop that turns a model into a harness —
  including the exit condition, the parser, and an independent verifier that
  decides what the loop is allowed to accept. Use when writing an agentic
  pipeline, or when a loop reports success on work nobody checked.
allowed-tools: Read, Grep, Glob
```

# The loop is yours; the model is a component in it

A harness is four moves — plan, act, verify, stop — and the security engineer
owns three of them. What the loop **accepts** is decided by the verifier, not by
the model, and a loop with no verifier accepts whatever came back and reports
success.

## When to use this

Whenever you are about to put a model in a for-loop: triage, remediation,
detection authoring, anything that iterates until it is satisfied.

## Procedure

**1 — Write the exit condition before the prompt.** A loop whose exit is easier
than the work will take the exit. Do not offer the model a way to declare
itself done; decide that from the outside, on the artefact.

**2 — Parse defensively, because parsing is the harness's job.** Asked for one
line, a small model returns the whole function and a larger one returns a fenced
block. Both are reasonable readings. Take what you need from what arrives rather
than requiring the model to do you a favour.

**3 — Write the verifier as an independent check.** Independent means it does
not ask the model whether the answer is good. It examines the artefact: does the
line use a placeholder, does the test pass, does the exploit stop working.

**4 — Test the verifier against a correct answer you did not expect.** This is
the step that gets skipped. A verifier that accepts only one spelling of correct
rejects real work and burns the budget doing it — check `%s`, `:name` and `$1`
before shipping a check for `?`.

**5 — Test it against a plausible wrong answer.** Something that reads like a
fix and is not — an escape function wrapped around concatenation. If the
verifier accepts it, the loop ships it, reports success, and the trace looks
clean.

## Example

**Input** — the fixture committed at the top of [`scripts/agentic_harness_loop.py`](scripts/agentic_harness_loop.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
the loop, wired: ['plan', 'act', 'verify', 'stop']
backend  : replay
steps    : 1
accepted : return DB.execute("SELECT * FROM bookings WHERE ref=?", (ref,))
verified : None   <- nothing checked it

The loop stopped because the model produced something, which is not the
same as producing something correct. Whatever came back was accepted.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "loop": {"moves": ["plan", "act", "verify", "stop"], "max_steps": 0, "exit_owned_by": "harness|model"},
  "parse": {"expected": "str", "shapes_seen": ["str"], "strategy": "str"},
  "verifier": {"independent": true, "accepts": ["str"], "rejects": ["str"]},
  "runs": [{"verifier": "none|present", "steps": 0, "accepted": "str", "verified": null}]
}
```

Report the unverified run too. The pair is the lesson: the same model and the
same prompt, with and without something checking the answer.

## Failure modes

- **Offering the model a DONE exit.** It will take it on the first turn.
- **A verifier that accepts one spelling.** It rejects correct work, repeatedly,
  and the budget goes on retries.
- **Asking a model to judge the answer.** That is a second opinion, not a
  verification.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/agentic-harness-loop/scripts/agentic_harness_loop.py
SCRIPT = "skills/appsec/agentic-harness-loop/scripts/agentic_harness_loop.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 4 · The part worth keeping

Read the last block of that output again. The answer it rejects —
`ref=" + escape(ref)` — is the one that matters: it reads like a fix, it would
pass a human skim, and it is still concatenation. Without a verifier the loop
accepts it, reports success and files a clean trace.

That is the whole reason this chapter defines the word before it builds
anything. Every stage after this one is a harness, and for each of them the
question is the same: **what, other than the model, decided that this worked?**

## What you just proved

The loop runs with a real model behind `ask()` — a labelled replay offline, a real open-weight call when one is served. With no verifier it accepts whatever came back and reports `verified: None`. With the verifier the same model and prompt produce an accepted, parameterised line; a narrow verifier that only accepts `?` is shown rejecting a correct psycopg fix, and a plausible answer wrapping the input in `escape()` is refused, because it is still concatenation.

## Your turn

Name your pipeline's verifier out loud. If the sentence contains "the model checks" or "it looks right", you have a judge, and a judge approves confident prose — including prose that contradicts the finding it is attached to.

---

**Next → [B2.2 · Threat modelling from what the estate already knows](https://spbreed.github.io/cyber-commons/lessons/B2.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*